In [2]:
# ============================================================
# CELL 1 : Install Required LangChain Packages
# ============================================================


# !pip install -q \
# langchain \
# langchain-community \
# langchain-core \
# langchain-google-genai \
# langchain-huggingface \
# langchain-chroma \
# chromadb \
# sentence-transformers \
# sqlalchemy \
# pymysql \
# python-dotenv \
# pandas \
# numpy \
# pypdf \
# beautifulsoup4 \
# requests \
# lxml



# ============================================================
# CELL 2 : Import Required Libraries
# ============================================================


import os
import warnings

from dotenv import load_dotenv


# LangChain Imports

from langchain_google_genai import ChatGoogleGenerativeAI

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.utilities import SQLDatabase

from langchain_chroma import Chroma


warnings.filterwarnings("ignore")


print("Libraries Imported Successfully")



c:\Users\Nikhil_\Desktop\RAGTRainingCRM\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\Nikhil_\AppData\Local\Temp\ipykernel_1324\669781583.py:44: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


Libraries Imported Successfully


In [3]:


# ============================================================
# CELL 3 : Load Environment Variables
# ============================================================


load_dotenv()


# Gemini

GOOGLE_API_KEY = os.getenv(
    "GOOGLE_API_KEY"
)


# MYSQL

MYSQL_HOST = os.getenv(
    "MYSQL_HOST"
)

MYSQL_PORT = os.getenv(
    "MYSQL_PORT"
)

MYSQL_USER = os.getenv(
    "MYSQL_USER"
)

MYSQL_PASSWORD = os.getenv(
    "MYSQL_PASSWORD"
)

MYSQL_DATABASE = os.getenv(
    "MYSQL_DATABASE"
)



# Chroma

CHROMA_DB_PATH = os.getenv(
    "CHROMA_DB_PATH"
)



print("Environment Variables Loaded")



Environment Variables Loaded


In [9]:


# ============================================================
# CELL 4 : Check Environment Configuration
# ============================================================


config = {

    "Gemini API":
    GOOGLE_API_KEY is not None,

    "MYSQL":
    MYSQL_DATABASE,

    "Chroma Path":
    CHROMA_DB_PATH

}


for key,value in config.items():

    print(
        key,
        ":",
        value
    )



# ============================================================
# CELL 5 : Initialize Gemini LLM using LangChain
# ============================================================


llm = ChatGoogleGenerativeAI(

    model="gemini-3.1-flash-lite",

    google_api_key=GOOGLE_API_KEY,

    temperature=0.3

)



print(
    "Gemini LangChain Model Initialized"
)




Gemini API : True
MYSQL : employee_crm
Chroma Path : None
Gemini LangChain Model Initialized


In [10]:

# ============================================================
# CELL 6 : Test Gemini LangChain Connection
# ============================================================


try:


    response = llm.invoke(

        "Explain CRM Agent in one sentence"

    )


    print(
        response.content
    )


except Exception as e:


    print(
        "Gemini Connection Error"
    )

    print(e)



# ============================================================
# CELL 7 : Create Project Directories
# ============================================================


folders = [

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/data",

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/data/documents",

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/data/website",

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/chroma_db",

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/memory",

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/logs",

    "C:/Users/Nikhil_/Desktop/RAGTRainingCRM/sql"

]


for folder in folders:


    os.makedirs(

        folder,

        exist_ok=True

    )


print(
    "Project Directories Created"
)




[{'type': 'text', 'text': 'A CRM agent is an AI-powered assistant that automates customer interactions and manages data within a Customer Relationship Management system to streamline support, sales, and personalized engagement.', 'extras': {'signature': 'EjQKMgERTTIPErrFU02/zhoCye6aKS1QVaLF89c7tCrs4TgAhKm+xdrI6seHsrjJX0OiAZJr'}}]
Project Directories Created


In [16]:

# ============================================================
# CELL 8 : Initialize LangChain Embedding Model
# ============================================================


embedding_model = HuggingFaceEmbeddings(

    model_name=
    "sentence-transformers/all-MiniLM-L6-v2"

)



print(
    "LangChain Embedding Model Loaded"
)



# ============================================================
# CELL 9 : Test Embedding Generation
# ============================================================


sample_document = [

    "Data Science course contains Python Machine Learning NLP"

]


embedding_result = embedding_model.embed_documents(

    sample_document

)



print(

    "Embedding Generated"

)


print(

    "Vector Size :",

    len(embedding_result[0])

)



OptionError: Option 'compute.use_bottleneck' has already been registered

In [ ]:


# ============================================================
# CELL 10 : Initialize Chroma Vector Database
# ============================================================


vector_db = Chroma(

    collection_name="crm_course_documents",

    embedding_function=embedding_model,

    persist_directory=CHROMA_DB_PATH

)



print(

    "ChromaDB Initialized Successfully"

)



# ============================================================
# CELL 11 : Test Chroma Document Storage
# ============================================================


documents = [

    "Python Full Stack Course",

    "Data Science Course",

    "Agentic AI Course"

]


vector_db.add_texts(

    documents

)


print(

    "Documents Stored in ChromaDB"

)



# ============================================================
# CELL 12 : Test Similarity Search
# ============================================================


result = vector_db.similarity_search(

    "AI course",

    k=2

)



for doc in result:

    print(

        doc.page_content

    )



# ============================================================
# CELL 13 : Initialize LangChain MySQL Connection
# ============================================================


mysql_uri = (

    f"mysql+pymysql://"

    f"{MYSQL_USER}:"

    f"{MYSQL_PASSWORD}@"

    f"{MYSQL_HOST}:"

    f"{MYSQL_PORT}/"

    f"{MYSQL_DATABASE}"

)



try:


    db = SQLDatabase.from_uri(

        mysql_uri

    )


    print(

        "MySQL Connected Using LangChain"

    )


except Exception as e:


    print(

        "MySQL Connection Failed"

    )

    print(e)



# ============================================================
# CELL 14 : Test Database Connection
# ============================================================


try:


    tables = db.get_usable_table_names()


    print(

        "Available Tables"

    )


    print(

        tables

    )


except Exception as e:


    print(e)



# ============================================================
# CELL 15 : Environment Validation Summary
# ============================================================


print(
"""
=================================================

CRM QUEST AGENT ENVIRONMENT READY

✔ LangChain Installed

✔ Gemini LLM Connected

✔ HuggingFace Embeddings Loaded

✔ Chroma Vector Database Ready

✔ MySQL LangChain Connection Ready

✔ Project Structure Created

=================================================
"""
)